In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
train=pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test=pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")
sample=pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv")



In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [4]:
# 1. Kolon Eşleme Sözlüğü (train_columns)
train_columns = {
    'Id': 'id',
    'MSSubClass': 'bina_sinifi',
    'MSZoning': 'imar_bolgesi',
    'LotFrontage': 'cephe_uzunlugu',
    'LotArea': 'arsa_alani',
    'Street': 'sokak_turu',
    'Alley': 'gecit_turu',
    'LotShape': 'arsa_sekli',
    'LandContour': 'arazi_duzlugu',
    'Utilities': 'altyapi',
    'LotConfig': 'parsel_konumu',
    'LandSlope': 'arazi_egimi',
    'Neighborhood': 'mahalle',
    'Condition1': 'cevre_durumu_1',
    'Condition2': 'cevre_durumu_2',
    'BldgType': 'konut_tipi',
    'HouseStyle': 'ev_mimarisi',
    'OverallQual': 'genel_kalite',
    'OverallCond': 'genel_durum',
    'YearBuilt': 'yapim_yili',
    'YearRemodAdd': 'tadilat_yili',
    'RoofStyle': 'cati_tipi',
    'RoofMatl': 'cati_malzemesi',
    'Exterior1st': 'dis_cephe_1',
    'Exterior2nd': 'dis_cephe_2',
    'MasVnrType': 'duvar_kaplama_tipi',
    'MasVnrArea': 'duvar_kaplama_alani',
    'ExterQual': 'dis_cephe_kalitesi',
    'ExterCond': 'dis_cephe_durumu',
    'Foundation': 'temel_tipi',
    'BsmtQual': 'bodrum_kalitesi',
    'BsmtCond': 'bodrum_durumu',
    'BsmtExposure': 'bodrum_isik_alma',
    'BsmtFinType1': 'bodrum_yasam_tipi_1',
    'BsmtFinSF1': 'bitmis_bodrum_alani_1',
    'BsmtFinType2': 'bodrum_yasam_tipi_2',
    'BsmtFinSF2': 'bitmis_bodrum_alani_2',
    'BsmtUnfSF': 'ham_bodrum_alani',
    'TotalBsmtSF': 'toplam_bodrum_alani',
    'Heating': 'isitma_tipi',
    'HeatingQC': 'isitma_kalitesi',
    'CentralAir': 'merkezi_klima',
    'Electrical': 'elektrik_sistemi',
    '1stFlrSF': 'birinci_kat_alani',
    '2ndFlrSF': 'ikinci_kat_alani',
    'LowQualFinSF': 'dusuk_kalite_alan',
    'GrLivArea': 'zemin_ustu_yasam_alani',
    'BsmtFullBath': 'bodrum_tam_banyo',
    'BsmtHalfBath': 'bodrum_yarim_banyo',
    'FullBath': 'tam_banyo',
    'HalfBath': 'yarim_banyo',
    'BedroomAbvGr': 'yatak_odasi_sayisi',
    'KitchenAbvGr': 'mutfak_sayisi',
    'KitchenQual': 'mutfak_kalitesi',
    'TotRmsAbvGrd': 'toplam_oda_sayisi',
    'Functional': 'islevsellik',
    'Fireplaces': 'somine_sayisi',
    'FireplaceQu': 'somine_kalitesi',
    'GarageType': 'garaj_tipi',
    'GarageYrBlt': 'garaj_yapim_yili',
    'GarageFinish': 'garaj_kaplama',
    'GarageCars': 'garaj_arac_kapasitesi',
    'GarageArea': 'garaj_alani',
    'GarageQual': 'garaj_kalitesi',
    'GarageCond': 'garaj_durumu',
    'PavedDrive': 'garaj_yolu_kaplamasi',
    'WoodDeckSF': 'ahsap_veranda_alani',
    'OpenPorchSF': 'acik_sundurma_alani',
    'EnclosedPorch': 'kapali_sundurma_alani',
    '3SsnPorch': 'uc_mevsim_sundurma_alani',
    'ScreenPorch': 'camli_balkon_alani',
    'PoolArea': 'havuz_alani',
    'PoolQC': 'havuz_kalitesi',
    'Fence': 'cit_kalitesi',
    'MiscFeature': 'ek_ozellik',
    'MiscVal': 'ek_ozellik_degeri',
    'MoSold': 'satildigi_ay',
    'YrSold': 'satildigi_yil',
    'SaleType': 'satis_turu',
    'SaleCondition': 'satis_kosulu',
    'SalePrice': 'satis_fiyati'
}

# 2. Hem Train hem Test Setinde Kolonları Değiştirme
train.rename(columns=train_columns, inplace=True)
test.rename(columns=train_columns, inplace=True)

# 3. Kontrol
print(f"Train kolon sayısı: {train.shape[1]}")
print(f"Test kolon sayısı: {test.shape[1]}")
print("\nTrain ilk 5 kolon:", list(train.columns[:5]))
print("Test ilk 5 kolon:", list(test.columns[:5]))

Train kolon sayısı: 81
Test kolon sayısı: 80

Train ilk 5 kolon: ['id', 'bina_sinifi', 'imar_bolgesi', 'cephe_uzunlugu', 'arsa_alani']
Test ilk 5 kolon: ['id', 'bina_sinifi', 'imar_bolgesi', 'cephe_uzunlugu', 'arsa_alani']


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        1460 non-null   int64  
 1   bina_sinifi               1460 non-null   int64  
 2   imar_bolgesi              1460 non-null   object 
 3   cephe_uzunlugu            1201 non-null   float64
 4   arsa_alani                1460 non-null   int64  
 5   sokak_turu                1460 non-null   object 
 6   gecit_turu                91 non-null     object 
 7   arsa_sekli                1460 non-null   object 
 8   arazi_duzlugu             1460 non-null   object 
 9   altyapi                   1460 non-null   object 
 10  parsel_konumu             1460 non-null   object 
 11  arazi_egimi               1460 non-null   object 
 12  mahalle                   1460 non-null   object 
 13  cevre_durumu_1            1460 non-null   object 
 14  cevre_du

In [6]:
for col in train.columns:
    if train[col].dtype == 'object':
        print(col)

imar_bolgesi
sokak_turu
gecit_turu
arsa_sekli
arazi_duzlugu
altyapi
parsel_konumu
arazi_egimi
mahalle
cevre_durumu_1
cevre_durumu_2
konut_tipi
ev_mimarisi
cati_tipi
cati_malzemesi
dis_cephe_1
dis_cephe_2
duvar_kaplama_tipi
dis_cephe_kalitesi
dis_cephe_durumu
temel_tipi
bodrum_kalitesi
bodrum_durumu
bodrum_isik_alma
bodrum_yasam_tipi_1
bodrum_yasam_tipi_2
isitma_tipi
isitma_kalitesi
merkezi_klima
elektrik_sistemi
mutfak_kalitesi
islevsellik
somine_kalitesi
garaj_tipi
garaj_kaplama
garaj_kalitesi
garaj_durumu
garaj_yolu_kaplamasi
havuz_kalitesi
cit_kalitesi
ek_ozellik
satis_turu
satis_kosulu
